<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V1_Source_Target.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Network Analysis – Cross-City Transfer Learning
**Ziel:** Source- und Target-Städte für CCTL identifizieren

**Wichtige Vorbedingung:** Die Demand-Daten enthalten verschiedene Zeilentypen je nach Station:

| Typ | Beispiel | Bedeutung | Verwendung |
|---|---|---|---|
| `real` | `Hauptbahnhof Süd` | Echter Stationsname | ✅ Kernanalyse |
| `virtual` | `Virtuell Mitte` | Virtuelle Zone, aber stationsgebunden | ✅ Kernanalyse |
| `bike` | `BIKE-12345` | Bike wurde **außerhalb** einer Station abgestellt (Free-Floating-Verhalten) | ❌ Ausschluss |
| `recording` | `recording_test` | Dummy/Testeinträge ohne realen Wert | ❌ Ausschluss |
| `only_nums` | `1234` | Kein echter Name, nicht zuordenbar | ⚠️ Prüfen |

> **Für die AD-Analyse zählen ausschließlich `real`-Stationen.**  
> `bike`-Zeilen repräsentieren kein stationsbasiertes Verhalten und würden die Nachfragemuster verzerren.

In [1]:
# Zugangsberechtigung auf Drive

from google.colab import drive
drive.mount('/content/drive')

# Gezippte Daten werden entpackt und in hohes Verzeichnis gelegt

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"

!unzip -q "/content/data.zip" -d "/content"

!rm "/content/data.zip"
!rm "/content/_MACOSX"


Mounted at /content/drive
Mounted at /content/drive
rm: cannot remove '/content/_MACOSX': No such file or directory
rm: cannot remove '/content/_MACOSX': No such file or directory


Aus letzten Veruschen sind folgende Städte bekannt:

[Heidelberg, Marburg, Gießen, Freiburg, Leverkusen, Braunschweig, Kaiserslautern, Dortmund, Kassel, Duisburg, Ludwigshafen, Bochum, Wiesbaden, Winsen (Luhe), Essen, Offenburg]


In [7]:
import pandas as pd
import numpy as np
import glob, os, re
import warnings
warnings.filterwarnings('ignore')

# ── Pfade ──────────────────────────────────────────────────────
DEMAND_BASE    = '/content/data/demand'
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'

# ── Kandidaten aus Schritt 1 (station_based) ───────────────────
CANDIDATES = [
    'Mannheim', 'Heidelberg', 'Marburg', 'Gießen', 'Freiburg',
    'Leverkusen', 'Braunschweig', 'Kaiserslautern', 'Dortmund',
    'Kassel', 'Duisburg', 'Ludwigshafen', 'Bochum', 'Wiesbaden',
    'Winsen (Luhe)', 'Essen', 'Offenburg',
]

# ── Schwellenwerte ─────────────────────────────────────────────
SOURCE_MIN_MONTHS   = 12
SOURCE_MIN_COVERAGE = 90.0
TARGET_MIN_MONTHS   = 3
TARGET_MIN_COVERAGE = 80.0
MAX_GAP_DAYS        = 30

# ── Mindest-EPD (Events Per Day) pro Station ───────────────────
# EPD = (n_lends + n_returns) einer Station, gemittelt über alle aktiven Tage.
# Bedeutung: Wie viele Ausleih- + Rückgabe-Ereignisse passieren im Schnitt
# pro Tag an einer Station?
# Warum wichtig für AD: Eine Station mit < 1 EPD liefert an den meisten Tagen
# den Wert 0 → kein auswertbares Signal, Anomalien nicht erkennbar.
MIN_EPD = 1.0

In [8]:
station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})

def classify_station(name: str) -> str:
    """Klassifiziert einen Stationsnamen in einen von 5 Typen."""
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names['station_type'] = station_names['station_name'].apply(classify_station)

# Lookup-Dict: station_name_id → station_type  (für schnellen Join)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()

print('Station-Typ-Verteilung in station_names:')
print(station_names['station_type'].value_counts().to_string())

Station-Typ-Verteilung in station_names:
station_type
bike         44747
real         13040
virtual       4827
recording     1972
only_nums       51


In [9]:
def load_city(city: str, base: str) -> pd.DataFrame | None:
    """Lädt alle Parquet-Dateien aus dem Stadtordner rekursiv."""
    pattern = os.path.join(base, city, '**', '*.parquet')
    files   = glob.glob(pattern, recursive=True)
    if not files:
        pattern = os.path.join(base, city, '*.parquet')
        files   = glob.glob(pattern)
    if not files:
        return None

    cols  = ['network_name', 'timestamp', 'station_id', 'station_name_id', 'n_lends', 'n_returns']
    parts = [pd.read_parquet(f, columns=cols) for f in files]
    return pd.concat(parts, ignore_index=True)


city_dfs_raw: dict[str, pd.DataFrame] = {}
missing = []

for city in CANDIDATES:
    df = load_city(city, DEMAND_BASE)
    if df is None:
        missing.append(city)
        print(f'  ⚠️  {city}: nicht gefunden')
        continue

    df['timestamp']     = pd.to_datetime(df['timestamp'], utc=True)
    df['date']          = df['timestamp'].dt.date

    # Station-Typ aus Lookup anhängen (kein teurer Merge nötig)
    df['station_type']  = df['station_name_id'].map(type_lookup).fillna('unknown')

    city_dfs_raw[city]  = df
    n_real = (df['station_type'] == 'real').sum()
    n_bike = (df['station_type'] == 'bike').sum()
    n_rec  = (df['station_type'] == 'recording').sum()
    print(f'  ✅ {city}: {len(df):>10,} Zeilen  '
          f'(real={n_real:,}  bike={n_bike:,}  recording={n_rec:,})')

print(f'\n→ {len(city_dfs_raw)} Städte geladen, {len(missing)} fehlen: {missing}')

  ✅ Mannheim:  2,683,584 Zeilen  (real=2,579,329  bike=103,497  recording=754)
  ✅ Heidelberg:  1,957,371 Zeilen  (real=1,873,100  bike=84,206  recording=65)
  ✅ Marburg:  1,908,545 Zeilen  (real=1,837,213  bike=71,332  recording=0)
  ✅ Gießen:  1,082,120 Zeilen  (real=1,041,980  bike=40,091  recording=49)
  ✅ Freiburg:    904,037 Zeilen  (real=872,531  bike=31,335  recording=171)
  ✅ Leverkusen:    631,820 Zeilen  (real=602,444  bike=29,145  recording=229)
  ✅ Braunschweig:    420,448 Zeilen  (real=399,470  bike=20,924  recording=32)
  ✅ Kaiserslautern:    376,263 Zeilen  (real=357,303  bike=18,820  recording=138)
  ✅ Dortmund:    345,749 Zeilen  (real=334,022  bike=11,471  recording=256)
  ✅ Kassel:    337,986 Zeilen  (real=323,835  bike=14,044  recording=106)
  ✅ Duisburg:    253,832 Zeilen  (real=246,265  bike=7,419  recording=148)
  ✅ Ludwigshafen:    243,156 Zeilen  (real=227,788  bike=15,309  recording=57)
  ✅ Bochum:    203,035 Zeilen  (real=195,168  bike=7,750  recording=117)


In [10]:
KEEP_TYPES = {'real'}   # ← nur echte Stationen; 'virtual' bei Bedarf ergänzen

city_dfs: dict[str, pd.DataFrame] = {}

print(f'{'Stadt':<20} {'Gesamt':>10} {'→ real':>10} {'Anteil':>8}  Verworfen')
print('-' * 65)

for city, df in city_dfs_raw.items():
    df_real  = df[df['station_type'].isin(KEEP_TYPES)].copy()
    n_before = len(df)
    n_after  = len(df_real)
    pct      = n_after / n_before * 100 if n_before > 0 else 0

    # Aufschlüsselung der verworfenen Zeilen
    dropped = df[~df['station_type'].isin(KEEP_TYPES)]['station_type'].value_counts().to_dict()

    city_dfs[city] = df_real
    print(f'{city:<20} {n_before:>10,} {n_after:>10,} {pct:>7.1f}%  {dropped}')

Stadt                    Gesamt     → real   Anteil  Verworfen
-----------------------------------------------------------------
Mannheim              2,683,584  2,579,329    96.1%  {'bike': 103497, 'recording': 754, 'only_nums': 4}
Heidelberg            1,957,371  1,873,100    95.7%  {'bike': 84206, 'recording': 65}
Marburg               1,908,545  1,837,213    96.3%  {'bike': 71332}
Gießen                1,082,120  1,041,980    96.3%  {'bike': 40091, 'recording': 49}
Freiburg                904,037    872,531    96.5%  {'bike': 31335, 'recording': 171}
Leverkusen              631,820    602,444    95.4%  {'bike': 29145, 'recording': 229, 'only_nums': 2}
Braunschweig            420,448    399,470    95.0%  {'bike': 20924, 'recording': 32, 'virtual': 22}
Kaiserslautern          376,263    357,303    95.0%  {'bike': 18820, 'recording': 138, 'only_nums': 2}
Dortmund                345,749    334,022    96.6%  {'bike': 11471, 'recording': 256}
Kassel                  337,986    323,835   

In [11]:
def temporal_coverage(city: str, df: pd.DataFrame) -> dict:
    min_date    = df['timestamp'].min()
    max_date    = df['timestamp'].max()
    total_days  = max(1, (max_date - min_date).days)
    active_days = df['date'].nunique()
    coverage    = active_days / total_days * 100
    months      = total_days / 30.44

    # Größte Lücke
    sorted_dates = sorted(df['date'].unique())
    gaps = [
        (pd.Timestamp(sorted_dates[i]) - pd.Timestamp(sorted_dates[i-1])).days - 1
        for i in range(1, len(sorted_dates))
    ]
    max_gap = max(gaps, default=0)

    if months >= SOURCE_MIN_MONTHS and coverage >= SOURCE_MIN_COVERAGE and max_gap <= MAX_GAP_DAYS:
        role = 'SOURCE'
    elif months >= TARGET_MIN_MONTHS and coverage >= TARGET_MIN_COVERAGE:
        role = 'TARGET'
    elif months >= TARGET_MIN_MONTHS:
        role = 'TARGET_LOW_COV'
    else:
        role = 'EXCLUDE'

    return dict(city=city, min_date=min_date.date(), max_date=max_date.date(),
                total_days=total_days, total_months=round(months, 1),
                active_days=active_days, coverage_pct=round(coverage, 1),
                max_gap_days=max_gap, role=role)


coverage_rows = [temporal_coverage(c, df) for c, df in city_dfs.items()]
df_coverage   = pd.DataFrame(coverage_rows).sort_values('total_months', ascending=False)

icon_map = {'SOURCE': '✅ SOURCE', 'TARGET': '✅ TARGET',
            'TARGET_LOW_COV': '⚠️ TARGET (Coverage)', 'EXCLUDE': '❌ EXCLUDE'}

display(
    df_coverage.assign(role=df_coverage['role'].map(icon_map))
    .style
    .format({'coverage_pct': '{:.1f}%', 'total_months': '{:.1f}'})
    .bar(subset=['coverage_pct'], color='#90EE90', vmin=0, vmax=100)
    .bar(subset=['max_gap_days'], color='#FFB6C1', vmin=0)
)

,city,min_date,max_date,total_days,total_months,active_days,coverage_pct,max_gap_days,role
0,Mannheim,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
1,Heidelberg,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
2,Marburg,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
3,Gießen,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
5,Leverkusen,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
7,Kaiserslautern,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
14,Winsen (Luhe),2023-04-01,2026-02-02,1038,34.1,1007,97.0%,20,✅ SOURCE
11,Ludwigshafen,2023-03-31,2026-02-02,1039,34.1,1039,100.0%,1,✅ SOURCE
9,Kassel,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE
13,Wiesbaden,2023-03-31,2026-02-02,1039,34.1,1008,97.0%,20,✅ SOURCE


In [12]:
def demand_volume(city: str, df: pd.DataFrame) -> dict:
    """
    Berechnet EPD-Kennzahlen ausschließlich auf Basis echter Stationen.
    df ist bereits auf station_type == 'real' gefiltert.
    """
    n_stations   = df['station_id'].nunique()
    n_days       = df['date'].nunique()
    total_events = df['n_lends'].sum() + df['n_returns'].sum()

    # Netzwerk-Durchschnitt (durch Hub-Stationen verzerrt!)
    avg_epd_network = total_events / n_stations / n_days if (n_stations * n_days) > 0 else 0

    # EPD je Station → robustere Verteilungskennzahlen
    per_station = (
        df.groupby('station_id')
        .apply(lambda x: (x['n_lends'].sum() + x['n_returns'].sum()) / n_days)
        .reset_index(name='epd')
    )

    # Anteil Stationen unterhalb Mindest-EPD
    # → hoher Anteil = viele Stationen zu inaktiv für AD
    low_signal_pct = (per_station['epd'] < MIN_EPD).mean() * 100

    median_epd = per_station['epd'].median()
    p25_epd    = per_station['epd'].quantile(0.25)   # 25% der Stationen haben EPD ≤ dieser Wert
    p75_epd    = per_station['epd'].quantile(0.75)

    # Signal gut, wenn: Median ≥ MIN_EPD UND weniger als 50% der Stationen zu schwach
    signal_ok  = median_epd >= MIN_EPD and low_signal_pct < 50

    return dict(
        city=city,
        n_real_stations=n_stations,
        avg_epd=round(avg_epd_network, 2),
        median_epd=round(median_epd, 2),
        p25_epd=round(p25_epd, 2),
        p75_epd=round(p75_epd, 2),
        low_signal_pct=round(low_signal_pct, 1),  # % Stationen mit EPD < MIN_EPD
        signal_ok=signal_ok,
    )


volume_rows = [demand_volume(c, df) for c, df in city_dfs.items()]
df_volume   = pd.DataFrame(volume_rows).sort_values('median_epd', ascending=False)

display(
    df_volume.assign(signal_ok=df_volume['signal_ok'].map({True: '✅ OK', False: '⚠️ SCHWACH'}))
    .style
    .format({'avg_epd': '{:.2f}', 'median_epd': '{:.2f}',
             'p25_epd': '{:.2f}', 'p75_epd': '{:.2f}', 'low_signal_pct': '{:.1f}%'})
    .bar(subset=['avg_epd', 'median_epd'], color='#ADD8E6')
    .bar(subset=['low_signal_pct'], color='#FFB6C1', vmin=0, vmax=100)
)

,city,n_real_stations,avg_epd,median_epd,p25_epd,p75_epd,low_signal_pct,signal_ok
2,Marburg,61,43.43,29.78,7.90,59.47,8.2%,✅ OK
1,Heidelberg,70,35.62,23.92,9.86,45.42,15.7%,✅ OK
4,Freiburg,132,31.05,21.74,6.80,38.85,9.1%,✅ OK
0,Mannheim,128,26.56,16.91,2.12,41.41,18.8%,✅ OK
3,Gießen,67,21.35,12.73,1.56,28.88,22.4%,✅ OK
8,Dortmund,100,14.36,9.96,3.74,17.65,7.0%,✅ OK
10,Duisburg,58,18.38,8.78,2.82,19.91,6.9%,✅ OK
14,Winsen (Luhe),22,8.59,7.01,2.81,10.04,9.1%,✅ OK
7,Kaiserslautern,37,11.27,6.76,0.80,17.94,29.7%,✅ OK
12,Bochum,95,8.67,4.97,2.38,12.39,9.5%,✅ OK


In [13]:
df_summary = df_coverage.merge(
    df_volume[['city', 'n_real_stations', 'avg_epd', 'median_epd', 'low_signal_pct', 'signal_ok']],
    on='city'
)

def recommend(row):
    if row['role'] == 'SOURCE' and row['signal_ok']:      return 'SOURCE'
    if row['role'] in ('TARGET', 'TARGET_LOW_COV') and row['signal_ok']: return 'TARGET'
    return 'EXCLUDE'

df_summary['recommendation'] = df_summary.apply(recommend, axis=1)

print('=' * 65)
for rec, label, icon in [
    ('SOURCE',  'Source-Städte  (≥12 Mo, Cov ≥90%, Signal OK)', '🟢'),
    ('TARGET',  'Target-Städte  (≥3 Mo,  Cov ≥80%, Signal OK)', '🔵'),
    ('EXCLUDE', 'Ausgeschlossen', '🔴'),
]:
    subset = df_summary[df_summary['recommendation'] == rec]
    print(f'\n{icon} {label} ({len(subset)}):')
    for _, r in subset.iterrows():
        warn = []
        if r['coverage_pct'] < SOURCE_MIN_COVERAGE: warn.append(f'Cov={r["coverage_pct"]:.0f}%')
        if r['max_gap_days'] > MAX_GAP_DAYS:         warn.append(f'Gap={r["max_gap_days"]}d')
        if not r['signal_ok']:                        warn.append(f'EPD↓')
        warn_str = '  ⚠️ ' + ', '.join(warn) if warn else ''
        print(f"   • {r['city']:<20s} | {r['total_months']:5.1f} Mo | "
              f"Cov {r['coverage_pct']:5.1f}% | "
              f"Median-EPD {r['median_epd']:.2f} | "
              f"{r['n_real_stations']} Stationen"
              f"{warn_str}")

print('\n' + '=' * 65)


🟢 Source-Städte  (≥12 Mo, Cov ≥90%, Signal OK) (11):
   • Mannheim             |  34.1 Mo | Cov 100.0% | Median-EPD 16.91 | 128 Stationen
   • Heidelberg           |  34.1 Mo | Cov 100.0% | Median-EPD 23.92 | 70 Stationen
   • Marburg              |  34.1 Mo | Cov  97.0% | Median-EPD 29.78 | 61 Stationen
   • Gießen               |  34.1 Mo | Cov  97.0% | Median-EPD 12.73 | 67 Stationen
   • Leverkusen           |  34.1 Mo | Cov  97.0% | Median-EPD 2.46 | 152 Stationen
   • Kaiserslautern       |  34.1 Mo | Cov 100.0% | Median-EPD 6.76 | 37 Stationen
   • Winsen (Luhe)        |  34.1 Mo | Cov  97.0% | Median-EPD 7.01 | 22 Stationen
   • Ludwigshafen         |  34.1 Mo | Cov 100.0% | Median-EPD 2.77 | 65 Stationen
   • Kassel               |  34.1 Mo | Cov  97.0% | Median-EPD 2.55 | 77 Stationen
   • Wiesbaden            |  34.1 Mo | Cov  97.0% | Median-EPD 4.84 | 36 Stationen
   • Offenburg            |  30.5 Mo | Cov  98.8% | Median-EPD 4.09 | 28 Stationen

🔵 Target-Städte  (≥3 Mo,  

In [14]:
df_summary.to_csv('network_analysis_results.csv', index=False)
print('Gespeichert: network_analysis_results.csv')
df_summary

Gespeichert: network_analysis_results.csv


,city,min_date,max_date,total_days,total_months,active_days,coverage_pct,max_gap_days,role,n_real_stations,avg_epd,median_epd,low_signal_pct,signal_ok,recommendation
0,Mannheim,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,128,26.56,16.91,18.8,True,SOURCE
1,Heidelberg,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,70,35.62,23.92,15.7,True,SOURCE
2,Marburg,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,61,43.43,29.78,8.2,True,SOURCE
3,Gießen,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,67,21.35,12.73,22.4,True,SOURCE
4,Leverkusen,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,152,4.79,2.46,32.2,True,SOURCE
5,Kaiserslautern,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,37,11.27,6.76,29.7,True,SOURCE
6,Winsen (Luhe),2023-04-01,2026-02-02,1038,34.1,1007,97.0,20,SOURCE,22,8.59,7.01,9.1,True,SOURCE
7,Ludwigshafen,2023-03-31,2026-02-02,1039,34.1,1039,100.0,1,SOURCE,65,4.02,2.77,24.6,True,SOURCE
8,Kassel,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,77,5.47,2.55,24.7,True,SOURCE
9,Wiesbaden,2023-03-31,2026-02-02,1039,34.1,1008,97.0,20,SOURCE,36,5.58,4.84,11.1,True,SOURCE


# Phase 2 · Stadtprofilierung — Feature-Vektoren

**Ziel:** Eine Feature-Matrix erstellen, die jede Kandidatenstadt als numerischen Vektor beschreibt.  
Diese Matrix ist die Basis für die **Ähnlichkeitsberechnung in Phase 3** (Source–Target-Matching).

**Warum?** Ohne systematische Ähnlichkeitsprüfung riskiert man *Negative Transfer*:  
ein Modell, das auf einer funktional anderen Stadt trainiert wurde, verschlechtert die AD-Leistung.

**Dimensionen:**
| # | Dimension | Datenquelle | Beschreibt |
|---|---|---|---|
| 2.1 | POI-Profil | OSM | Funktionale Zusammensetzung des Stationsumfelds |
| 2.2 | Landnutzungsprofil | OSM_LANDUSE | Urbane Morphologie |
| 2.3 | Nachfrageprofil | DEMAND | Zeitliche Nutzungsmuster |
| 2.4 | Kontext | GEO_INFORMATION | Bundesland, geographische Lage |

In [28]:
import pandas as pd
import numpy as np
import glob, os, re
import geopandas as gpd
import pyarrow.parquet as pq
from shapely import wkb
import warnings
import geometry

!pip install fastparquet -q


warnings.filterwarnings('ignore')

# ── Pfade ──────────────────────────────────────────────────────────────────
DEMAND_BASE        = '/content/data/demand'          # Rohdaten, wie in network_analysis_v2
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'
OSM_PATH           = '/content/data/osm/osm.parquet'
OSM_LANDUSE_PATH   = '/content/data/osm_landuse/osm_landuse.parquet'
GEO_INFO_PATH      = '/content/data/geo_information/geo_information.parquet'

# ── Kandidaten aus network_analysis_v2-Ausgabe ─────────────────────────────
# Ausgabedatei von network_analysis_v2.ipynb
network_results = pd.read_csv('/content/network_analysis_results.csv')
CANDIDATES = network_results[
    network_results['recommendation'] != 'EXCLUDE'
]['city'].tolist()

print(f'Kandidaten-Netzwerke ({len(CANDIDATES)}): {CANDIDATES}')

Kandidaten-Netzwerke (17): ['Mannheim', 'Heidelberg', 'Marburg', 'Gießen', 'Leverkusen', 'Kaiserslautern', 'Winsen (Luhe)', 'Ludwigshafen', 'Kassel', 'Wiesbaden', 'Offenburg', 'Braunschweig', 'Freiburg', 'Dortmund', 'Duisburg', 'Bochum', 'Essen']


In [30]:
# ── Station Names laden & Typ-Klassifikation (wie in network_analysis_v2) ──
def classify_station(name: str) -> str:
    if not isinstance(name, str) or name.strip() == '': return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names['station_type'] = station_names['station_name'].apply(classify_station)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()
print(f'Station-Types geladen: {station_names["station_type"].value_counts().to_dict()}')

# ── DEMAND: pro Stadt laden, auf real-Stationen filtern ────────────────────
def load_city(city, base):
    files = glob.glob(os.path.join(base, city, '**', '*.parquet'), recursive=True)
    if not files:
        files = glob.glob(os.path.join(base, city, '*.parquet'))
    if not files: return None
    cols = ['network_name', 'timestamp', 'station_id', 'station_name_id', 'location_id', 'n_lends', 'n_returns']
    return pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)

parts = []
for city in CANDIDATES:
    df = load_city(city, DEMAND_BASE)
    if df is None:
        print(f'  ⚠️  {city}: nicht gefunden'); continue
    df['timestamp']    = pd.to_datetime(df['timestamp'], utc=True)
    df['station_type'] = df['station_name_id'].map(type_lookup).fillna('unknown')
    df_real = df[df['station_type'] == 'real']
    parts.append(df_real)
    print(f'  ✅ {city}: {len(df_real):>8,} real-Zeilen (von {len(df):,} gesamt)')

demand = pd.concat(parts, ignore_index=True)
demand['date'] = demand['timestamp'].dt.date
print(f'\nDEMAND gesamt: {len(demand):,} Zeilen | {demand["network_name"].nunique()} Netzwerke')

# ── Restliche Datenquellen ───────────────────────────────────────────────────

# GEO_INFO: location-Spalte enthält WKB-Geometrie → lat/lon extrahieren
geo_info = pq.read_table(GEO_INFO_PATH).to_pandas()
geom = geo_info['location'].apply(lambda x: wkb.loads(bytes.fromhex(x)))
geo_info['latitude']  = geom.apply(lambda g: round(g.y, 6))
geo_info['longitude'] = geom.apply(lambda g: round(g.x, 6))
geo_info = geo_info.drop(columns=['location'])
print(f'GEO_INFO:     {len(geo_info):>10,} Zeilen  |  Spalten: {list(geo_info.columns)}')

# OSM: direkt lesbar
osm = pd.read_parquet(OSM_PATH)
print(f'OSM:          {len(osm):>10,} Zeilen  |  Kategorien: {osm["entity_name"].nunique()}')

# OSM_LANDUSE: geometry (WKB) laden + Fläche berechnen
# Rohdaten haben keine area_m2/area_km2 → hier berechnen
def load_landuse(path: str) -> gpd.GeoDataFrame:
    df_raw = pd.read_parquet(path, engine='fastparquet')

    # Geometrie-Spalte deserialisieren (WKB bytes)
    df_raw['area'] = df_raw['area'].apply(lambda x: wkb.loads(x))

    # GeoDataFrame erstellen (Quell-CRS: WGS84)
    gdf = gpd.GeoDataFrame(df_raw, geometry='area', crs='EPSG:4326')

    # Für Flächenberechnung in Metrische CRS projizieren
    gdf_proj = gdf.to_crs('EPSG:3857')
    gdf['area_m2']  = gdf_proj['area'].area.round(2)
    gdf['area_km2'] = (gdf['area_m2'] / 1_000_000).round(6)

    # Geometrie als WKT speichern (für spätere Kompatibilität)
    # Achtung: Geometry-Spalte heißt 'area' (nicht 'geometry'), da so beim GeoDataFrame gesetzt
    gdf['area'] = gdf['area'].apply(lambda g: g.wkt)

    return gdf   # ← return nicht vergessen!

osm_landuse = load_landuse(OSM_LANDUSE_PATH)
print(f'OSM_LANDUSE:  {len(osm_landuse):>10,} Zeilen  |  Typen: {osm_landuse["landuse"].nunique()}')
print(f'              Spalten: {list(osm_landuse.columns)}')

Station-Types geladen: {'bike': 44747, 'real': 13040, 'virtual': 4827, 'recording': 1972, 'only_nums': 51}
  ✅ Mannheim: 2,579,329 real-Zeilen (von 2,683,584 gesamt)
  ✅ Heidelberg: 1,873,100 real-Zeilen (von 1,957,371 gesamt)
  ✅ Marburg: 1,837,213 real-Zeilen (von 1,908,545 gesamt)
  ✅ Gießen: 1,041,980 real-Zeilen (von 1,082,120 gesamt)
  ✅ Leverkusen:  602,444 real-Zeilen (von 631,820 gesamt)
  ✅ Kaiserslautern:  357,303 real-Zeilen (von 376,263 gesamt)
  ✅ Winsen (Luhe):  156,438 real-Zeilen (von 159,763 gesamt)
  ✅ Ludwigshafen:  227,788 real-Zeilen (von 243,156 gesamt)
  ✅ Kassel:  323,835 real-Zeilen (von 337,986 gesamt)
  ✅ Wiesbaden:  152,302 real-Zeilen (von 163,261 gesamt)
  ✅ Offenburg:  143,369 real-Zeilen (von 151,580 gesamt)
  ✅ Braunschweig:  399,470 real-Zeilen (von 420,448 gesamt)
  ✅ Freiburg:  872,531 real-Zeilen (von 904,037 gesamt)
  ✅ Dortmund:  334,022 real-Zeilen (von 345,749 gesamt)
  ✅ Duisburg:  246,265 real-Zeilen (von 253,832 gesamt)
  ✅ Bochum:  195,168 

In [31]:
# location_id → network_name Mapping aus Demand
loc_to_network = (
    demand[['location_id', 'network_name']]
    .drop_duplicates()
)

# OSM mit Netzwerk verknüpfen
osm_mapped = (
    osm
    .merge(loc_to_network, on='location_id', how='inner')   # nur Kandidaten-Standorte
)

print(f'OSM nach Filter auf Kandidaten: {len(osm_mapped):,} Einträge')
print(f'Vorhandene entity_name-Kategorien:\n{osm_mapped["entity_name"].value_counts().head(15).to_string()}')

OSM nach Filter auf Kandidaten: 1,302 Einträge
Vorhandene entity_name-Kategorien:
entity_name
restaurant     247
tram_stop      213
fast_food      174
cafe           141
clothes        115
dentist         71
pharmacy        60
library         53
supermarket     45
bank            41
bar             35
station         25
marketplace     20
university      19
school           8


In [32]:
# ── Absolute Häufigkeiten ───────────────────────────────────────────────────
poi_profile_abs = (
    osm_mapped
    .groupby(['network_name', 'entity_name'])
    .size()
    .unstack(fill_value=0)
)

# ── Normalisieren → Anteile (Summe pro Stadt = 1.0) ────────────────────────
# Warum Anteile? Große Städte haben mehr POIs absolut — Anteile machen Städte vergleichbar.
poi_profile_norm = poi_profile_abs.div(poi_profile_abs.sum(axis=1), axis=0)

# Spalten-Präfix für spätere Feature-Matrix
poi_profile_norm.columns = ['poi_' + c for c in poi_profile_norm.columns]

print(f'POI-Feature-Vektor: {poi_profile_norm.shape[0]} Städte × {poi_profile_norm.shape[1]} POI-Kategorien')
poi_profile_norm.round(3)

POI-Feature-Vektor: 17 Städte × 21 POI-Kategorien


,poi_bank,poi_bar,poi_cafe,poi_cinema,poi_clothes,poi_college,poi_dentist,poi_department_store,poi_fast_food,poi_library,...,poi_pharmacy,poi_place_of_worship,poi_restaurant,poi_school,poi_station,poi_supermarket,poi_theatre,poi_townhall,poi_tram_stop,poi_university
network_name,,,,,,,,,,,,,,,,,,,,,
Bochum,0.041,0.020,0.133,0.000,0.041,0.000,0.061,0.000,0.082,0.163,...,0.031,0.010,0.163,0.010,0.020,0.051,0.000,0.00,0.133,0.010
Braunschweig,0.028,0.009,0.084,0.000,0.131,0.000,0.075,0.000,0.093,0.019,...,0.037,0.000,0.159,0.000,0.000,0.028,0.009,0.00,0.290,0.019
Dortmund,0.050,0.000,0.062,0.000,0.088,0.012,0.062,0.012,0.138,0.025,...,0.038,0.000,0.225,0.000,0.038,0.038,0.000,0.00,0.200,0.012
Duisburg,0.033,0.000,0.033,0.033,0.000,0.000,0.067,0.000,0.167,0.000,...,0.133,0.000,0.133,0.000,0.000,0.033,0.000,0.10,0.267,0.000
Essen,0.035,0.026,0.087,0.009,0.113,0.000,0.052,0.000,0.165,0.000,...,0.078,0.000,0.165,0.000,0.052,0.009,0.000,0.00,0.191,0.000
Freiburg,0.024,0.016,0.111,0.008,0.032,0.008,0.056,0.000,0.087,0.008,...,0.048,0.008,0.175,0.000,0.000,0.079,0.000,0.00,0.325,0.000
Gießen,0.000,0.056,0.222,0.000,0.139,0.000,0.056,0.000,0.111,0.028,...,0.000,0.000,0.306,0.000,0.028,0.028,0.000,0.00,0.000,0.000
Heidelberg,0.039,0.013,0.130,0.000,0.117,0.000,0.117,0.000,0.143,0.000,...,0.052,0.000,0.182,0.000,0.026,0.039,0.000,0.00,0.130,0.000
Kaiserslautern,0.000,0.019,0.245,0.000,0.208,0.019,0.000,0.019,0.226,0.019,...,0.075,0.000,0.132,0.000,0.019,0.019,0.000,0.00,0.000,0.000


In [33]:
# ── city-Name in OSM_LANDUSE auf network_name mappen ───────────────────────
# Falls die Namen nicht 1:1 übereinstimmen, hier manuell anpassen:
city_name_map = {
    c: c for c in CANDIDATES   # Identitäts-Mapping als Ausgangspunkt
    # Beispiel für Abweichungen:
    # 'Winsen (Luhe)': 'Winsen',
}

# Verfügbare Städte in OSM_LANDUSE prüfen
available_cities = osm_landuse['city'].unique()
not_matched = [c for c in CANDIDATES if c not in available_cities]
if not_matched:
    print(f'⚠️  Folgende Netzwerke nicht in OSM_LANDUSE gefunden: {not_matched}')
    print('   → city_name_map oben anpassen!')

# Umgekehrtes Mapping für den Filter
reverse_map = {v: k for k, v in city_name_map.items()}
landuse_filtered = osm_landuse[osm_landuse['city'].isin(city_name_map.values())].copy()
landuse_filtered['network_name'] = landuse_filtered['city'].map(reverse_map)

In [34]:
# ── Flächen aggregieren ─────────────────────────────────────────────────────
landuse_profile_abs = (
    landuse_filtered
    .groupby(['network_name', 'landuse'])['area_km2']
    .sum()
    .unstack(fill_value=0)
)

# ── Normalisieren → Flächenanteile ─────────────────────────────────────────
# Warum Flächenanteile? Absoluter Flächenwert hängt von Stadtgröße ab;
# der Anteil z.B. 'residential' beschreibt Charakter unabhängig von Größe.
landuse_profile_norm = landuse_profile_abs.div(landuse_profile_abs.sum(axis=1), axis=0)
landuse_profile_norm.columns = ['lu_' + c for c in landuse_profile_norm.columns]

print(f'Landnutzungs-Feature-Vektor: {landuse_profile_norm.shape[0]} Städte × {landuse_profile_norm.shape[1]} Landnutzungstypen')
print(f'Landnutzungstypen: {list(landuse_profile_norm.columns)}')
landuse_profile_norm.round(3)

Landnutzungs-Feature-Vektor: 17 Städte × 65 Landnutzungstypen
Landnutzungstypen: ['lu_administration', 'lu_allotments', 'lu_animal_keeping', 'lu_apiary', 'lu_basin', 'lu_brownfield', 'lu_brownland', 'lu_cemetery', 'lu_civic_admin', 'lu_commercial', 'lu_construction', 'lu_depot', 'lu_education', 'lu_emergency', 'lu_fairground', 'lu_farmland', 'lu_farmyard', 'lu_flowerbed', 'lu_forest', 'lu_garages', 'lu_garden', 'lu_grass', 'lu_greenery', 'lu_greenfield', 'lu_greenhouse_horticulture', 'lu_harbour', 'lu_hedge', 'lu_highway', 'lu_historic', 'lu_hydro', 'lu_industrial', 'lu_institutional', 'lu_landfill', 'lu_locality', 'lu_logistics', 'lu_meadow', 'lu_military', 'lu_nature', 'lu_orchard', 'lu_paddock', 'lu_park', 'lu_plant_nursery', 'lu_plantation', 'lu_proposed', 'lu_quarry', 'lu_railway', 'lu_recreation_ground', 'lu_religious', 'lu_reservoir', 'lu_residential', 'lu_retail', 'lu_sand', 'lu_schoolyard', 'lu_scrub', 'lu_shelter', 'lu_sports', 'lu_storage', 'lu_street', 'lu_traffic_green', '

,lu_administration,lu_allotments,lu_animal_keeping,lu_apiary,lu_basin,lu_brownfield,lu_brownland,lu_cemetery,lu_civic_admin,lu_commercial,...,lu_sports,lu_storage,lu_street,lu_traffic_green,lu_traffic_island,lu_village_green,lu_vineyard,lu_wholesale,lu_winter_sports,lu_yes
network_name,,,,,,,,,,,,,,,,,,,,,
Bochum,0.0,0.024,0.0,0.0,0.001,0.003,0.0,0.020,0.000,0.032,...,0.0,0.0,0.0,0.0,0.0,0.001,0.000,0.0,0.000,0.0
Braunschweig,0.0,0.024,0.0,0.0,0.000,0.001,0.0,0.006,0.000,0.070,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
Dortmund,0.0,0.022,0.0,0.0,0.000,0.010,0.0,0.015,0.000,0.031,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
Duisburg,0.0,0.015,0.0,0.0,0.001,0.006,0.0,0.015,0.000,0.027,...,0.0,0.0,0.0,0.0,0.0,0.001,0.000,0.0,0.000,0.0
Essen,0.0,0.021,0.0,0.0,0.001,0.004,0.0,0.015,0.000,0.044,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
Freiburg,0.0,0.009,0.0,0.0,0.000,0.001,0.0,0.003,0.000,0.018,...,0.0,0.0,0.0,0.0,0.0,0.000,0.054,0.0,0.000,0.0
Gießen,0.0,0.014,0.0,0.0,0.000,0.004,0.0,0.005,0.002,0.007,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
Heidelberg,0.0,0.020,0.0,0.0,0.000,0.001,0.0,0.002,0.000,0.015,...,0.0,0.0,0.0,0.0,0.0,0.000,0.003,0.0,0.000,0.0
Kaiserslautern,0.0,0.002,0.0,0.0,0.000,0.001,0.0,0.002,0.000,0.011,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0


In [35]:
# ── Zeitfeatures extrahieren ────────────────────────────────────────────────
demand['hour']      = demand['timestamp'].dt.hour
demand['dayofweek'] = demand['timestamp'].dt.dayofweek   # 0=Mo, 6=So
demand['is_weekend']= demand['dayofweek'] >= 5

# ── A) Stündliches Profil (24h-Kurve) ──────────────────────────────────────
# Mittlere Ausleih- + Rückgabeereignisse pro Stunde, gemittelt über alle Stationen und Tage
hourly_profile = (
    demand
    .groupby(['network_name', 'hour'])[['n_lends', 'n_returns']]
    .mean()
    .unstack('hour')
)
# Spalten flach machen: (n_lends, 0) → 'h_lends_00', etc.
hourly_profile.columns = [
    f'h_{metric}_{hour:02d}' for metric, hour in hourly_profile.columns
]

# Normalisieren: relative Verteilung über den Tag (Summe = 1 pro Stadt)
# → macht Städte mit unterschiedlichem Gesamtvolumen vergleichbar
lends_cols   = [c for c in hourly_profile.columns if 'lends' in c]
returns_cols = [c for c in hourly_profile.columns if 'returns' in c]
hourly_norm  = hourly_profile.copy()
hourly_norm[lends_cols]   = hourly_norm[lends_cols].div(hourly_norm[lends_cols].sum(axis=1),   axis=0)
hourly_norm[returns_cols] = hourly_norm[returns_cols].div(hourly_norm[returns_cols].sum(axis=1), axis=0)

print(f'Stündliches Profil: {hourly_norm.shape}')

Stündliches Profil: (17, 48)


In [36]:
# ── B) Wöchentliches Profil (Mo–So) ────────────────────────────────────────
weekly_profile = (
    demand
    .groupby(['network_name', 'dayofweek'])[['n_lends', 'n_returns']]
    .mean()
    .unstack('dayofweek')
)
day_labels = {0:'Mo', 1:'Di', 2:'Mi', 3:'Do', 4:'Fr', 5:'Sa', 6:'So'}
weekly_profile.columns = [
    f'w_{metric}_{day_labels[day]}' for metric, day in weekly_profile.columns
]

# Normalisieren über die Woche
lends_w   = [c for c in weekly_profile.columns if 'lends' in c]
returns_w = [c for c in weekly_profile.columns if 'returns' in c]
weekly_norm = weekly_profile.copy()
weekly_norm[lends_w]   = weekly_norm[lends_w].div(weekly_norm[lends_w].sum(axis=1),   axis=0)
weekly_norm[returns_w] = weekly_norm[returns_w].div(weekly_norm[returns_w].sum(axis=1), axis=0)

print(f'Wöchentliches Profil: {weekly_norm.shape}')

Wöchentliches Profil: (17, 14)


In [37]:
# ── C) Commuter Ratio ───────────────────────────────────────────────────────
# Verhältnis: mittlere Wochentags-Nachfrage / mittlere Wochenend-Nachfrage
#
# Interpretation:
#   ratio >> 1  → Pendler-Stadt: hohe Nutzung Mo–Fr (Arbeitswege)
#   ratio ~= 1  → gleichmäßige Nutzung (Tourismus, Mischnutzung)
#   ratio < 1   → Freizeit-Stadt: mehr Nutzung am Wochenende
#
# Warum relevant für Transfer? Ein Pendler-System hat andere Anomalie-Muster
# als ein Freizeitrad-System → Transfer nur sinnvoll zwischen ähnlichen Typen.

weekday = demand[~demand['is_weekend']].groupby('network_name')['n_lends'].mean()
weekend = demand[demand['is_weekend']].groupby('network_name')['n_lends'].mean()
commuter_ratio = (weekday / weekend).rename('commuter_ratio').to_frame()

print('Commuter Ratio pro Netzwerk (Wochentag-Ø / Wochenende-Ø):')
print(commuter_ratio.sort_values('commuter_ratio', ascending=False).round(3).to_string())

Commuter Ratio pro Netzwerk (Wochentag-Ø / Wochenende-Ø):
                commuter_ratio
network_name                  
Wiesbaden                1.109
Kassel                   1.081
Essen                    1.078
Mannheim                 1.058
Gießen                   1.051
Marburg                  1.045
Offenburg                1.044
Kaiserslautern           1.032
Heidelberg               1.031
Braunschweig             1.029
Dortmund                 1.027
Duisburg                 1.005
Ludwigshafen             1.004
Bochum                   0.994
Freiburg                 0.993
Winsen (Luhe)            0.988
Leverkusen               0.958


In [43]:
# ── 2.4 · Kontextuelle Features ────────────────────────────────────────────

# Einwohnerzahlen manuell recherchiert (Quelle: Statistisches Bundesamt / Wikipedia, Stand 2024-2025)
population_map = {
    'Mannheim':        318035,
    'Heidelberg':      155756,
    'Marburg':          73544,
    'Gießen':           89179,
    'Freiburg':        237460,
    'Leverkusen':      168581,
    'Braunschweig':    252962,
    'Kaiserslautern':   101228,
    'Dortmund':        603462,
    'Kassel':          197230,
    'Duisburg':        502270,
    'Ludwigshafen':    177222,
    'Bochum':          358676,
    'Wiesbaden':       288850,
    'Winsen (Luhe)':    36961,
    'Essen':           574682,
    'Offenburg':        62993,
}

# location_id → Bundesland, Koordinaten aus GEO_INFO
geo_per_network = (
    demand[['network_name', 'location_id']]
    .drop_duplicates()
    .merge(geo_info[['location_id', 'federal_state_name', 'latitude', 'longitude', 'elevation']],
           on='location_id', how='left')
)

# Bundesland: Modus pro Netzwerk
federal_state = (
    geo_per_network
    .groupby('network_name')['federal_state_name']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'unknown')
    .rename('federal_state')
)

# Koordinaten + Höhe: Median pro Netzwerk
geo_stats = (
    geo_per_network
    .groupby('network_name')[['latitude', 'longitude']]
    .median()
    .round(3)
)

context_features = geo_stats.join(federal_state)
context_features['population'] = context_features.index.map(population_map)

# One-Hot-Encoding Bundesland (für numerische Feature-Matrix)
federal_dummies = pd.get_dummies(context_features['federal_state'], prefix='state')
context_numeric = context_features[['latitude', 'longitude', 'population']].join(federal_dummies)

print(f'Kontextuelle Features: {context_numeric.shape}')
context_features   # Anzeige mit Klartext-Bundesland

Kontextuelle Features: (17, 8)


,latitude,longitude,federal_state,population
network_name,,,,
Bochum,51.468,7.246,Nordrhein-Westfalen,358676
Braunschweig,52.264,10.526,Niedersachsen,252962
Dortmund,51.508,7.460,Nordrhein-Westfalen,603462
Duisburg,51.430,6.767,Nordrhein-Westfalen,502270
Essen,51.447,7.007,Nordrhein-Westfalen,574682
Freiburg,47.997,7.834,Baden-Württemberg,237460
Gießen,50.584,8.678,Hessen,89179
Heidelberg,49.409,8.677,Baden-Württemberg,155756
Kaiserslautern,49.442,7.760,Rheinland-Pfalz,101228


In [44]:
feature_blocks = {
    'POI':       poi_profile_norm,
    'Landuse':   landuse_profile_norm,
    'Hourly':    hourly_norm,
    'Weekly':    weekly_norm,
    'Commuter':  commuter_ratio,
    'Context':   context_numeric,
}

feature_matrix = pd.concat(feature_blocks.values(), axis=1).fillna(0)

# Übersicht: wie viele Features kommen aus welchem Block?
print('Feature-Matrix Zusammensetzung:')
print(f"{'Block':<12} {'Features':>8}")
print('-' * 22)
for name, df in feature_blocks.items():
    print(f'{name:<12} {df.shape[1]:>8}')
print('-' * 22)
print(f'{"GESAMT":<12} {feature_matrix.shape[1]:>8}')
print(f'\nFeature-Matrix: {feature_matrix.shape[0]} Städte × {feature_matrix.shape[1]} Features')

Feature-Matrix Zusammensetzung:
Block        Features
----------------------
POI                21
Landuse            65
Hourly             48
Weekly             14
Commuter            1
Context             8
----------------------
GESAMT            157

Feature-Matrix: 17 Städte × 157 Features


In [45]:
# ── Fehlende Werte prüfen ───────────────────────────────────────────────────
missing_pct = feature_matrix.isnull().mean() * 100
if missing_pct.max() > 0:
    print('⚠️  Spalten mit fehlenden Werten (nach fillna=0 sollte das 0 sein):')
    print(missing_pct[missing_pct > 0].to_string())
else:
    print('✅ Keine fehlenden Werte')

# ── Städte mit vollständigem Profil ────────────────────────────────────────
n_zero_rows = (feature_matrix.sum(axis=1) == 0).sum()
if n_zero_rows > 0:
    print(f'⚠️  {n_zero_rows} Städte mit Null-Vektor (komplett leer):')
    print(feature_matrix[feature_matrix.sum(axis=1) == 0].index.tolist())
else:
    print('✅ Alle Städte haben mindestens einen Non-Zero-Feature')

# ── Wertebereich-Check ──────────────────────────────────────────────────────
print('\nWertebereich der Feature-Blöcke:')
offset = 0
for name, df in feature_blocks.items():
    cols  = feature_matrix.columns[offset:offset + df.shape[1]]
    chunk = feature_matrix[cols]
    print(f'  {name:<12}  min={chunk.min().min():.3f}  max={chunk.max().max():.3f}  '
          f'mean={chunk.mean().mean():.3f}')
    offset += df.shape[1]

✅ Keine fehlenden Werte
✅ Alle Städte haben mindestens einen Non-Zero-Feature

Wertebereich der Feature-Blöcke:
  POI           min=0.000  max=0.444  mean=0.048
  Landuse       min=0.000  max=0.883  mean=0.015
  Hourly        min=0.027  max=0.077  mean=0.042
  Weekly        min=0.128  max=0.153  mean=0.143
  Commuter      min=0.958  max=1.109  mean=1.031
  Context       min=0.000  max=603462.000  mean=30883.148


In [46]:
# Haupt-Feature-Matrix
feature_matrix.to_parquet('city_feature_matrix.parquet')
feature_matrix.to_csv('city_feature_matrix.csv')

# Einzelne Blöcke (für Ableitungen in Phase 3 nützlich)
poi_profile_norm.to_parquet('features_poi.parquet')
landuse_profile_norm.to_parquet('features_landuse.parquet')
hourly_norm.to_parquet('features_hourly.parquet')
weekly_norm.to_parquet('features_weekly.parquet')
commuter_ratio.to_parquet('features_commuter.parquet')
context_features.to_csv('features_context.csv')   # mit Klartext-Bundesland für Doku

print('Gespeichert:')
print('  → city_feature_matrix.parquet/.csv   (Gesamt-Feature-Matrix, Input für Phase 3)')
print('  → features_poi/landuse/hourly/weekly/commuter/context.*   (Einzelblöcke)')
print(f'\nFinale Feature-Matrix: {feature_matrix.shape[0]} Städte × {feature_matrix.shape[1]} Features')
feature_matrix.round(4)

Gespeichert:
  → city_feature_matrix.parquet/.csv   (Gesamt-Feature-Matrix, Input für Phase 3)
  → features_poi/landuse/hourly/weekly/commuter/context.*   (Einzelblöcke)

Finale Feature-Matrix: 17 Städte × 157 Features


,poi_bank,poi_bar,poi_cafe,poi_cinema,poi_clothes,poi_college,poi_dentist,poi_department_store,poi_fast_food,poi_library,...,w_n_returns_So,commuter_ratio,latitude,longitude,population,state_Baden-Württemberg,state_Hessen,state_Niedersachsen,state_Nordrhein-Westfalen,state_Rheinland-Pfalz
network_name,,,,,,,,,,,,,,,,,,,,,
Bochum,0.0408,0.0204,0.1327,0.0000,0.0408,0.0000,0.0612,0.0000,0.0816,0.1633,...,0.1337,0.9937,51.468,7.246,358676,False,False,False,True,False
Braunschweig,0.0280,0.0093,0.0841,0.0000,0.1308,0.0000,0.0748,0.0000,0.0935,0.0187,...,0.1394,1.0291,52.264,10.526,252962,False,False,True,False,False
Dortmund,0.0500,0.0000,0.0625,0.0000,0.0875,0.0125,0.0625,0.0125,0.1375,0.0250,...,0.1389,1.0267,51.508,7.460,603462,False,False,False,True,False
Duisburg,0.0333,0.0000,0.0333,0.0333,0.0000,0.0000,0.0667,0.0000,0.1667,0.0000,...,0.1398,1.0048,51.430,6.767,502270,False,False,False,True,False
Essen,0.0348,0.0261,0.0870,0.0087,0.1130,0.0000,0.0522,0.0000,0.1652,0.0000,...,0.1332,1.0781,51.447,7.007,574682,False,False,False,True,False
Freiburg,0.0238,0.0159,0.1111,0.0079,0.0317,0.0079,0.0556,0.0000,0.0873,0.0079,...,0.1428,0.9931,47.997,7.834,237460,True,False,False,False,False
Gießen,0.0000,0.0556,0.2222,0.0000,0.1389,0.0000,0.0556,0.0000,0.1111,0.0278,...,0.1356,1.0513,50.584,8.678,89179,False,True,False,False,False
Heidelberg,0.0390,0.0130,0.1299,0.0000,0.1169,0.0000,0.1169,0.0000,0.1429,0.0000,...,0.1369,1.0305,49.409,8.677,155756,True,False,False,False,False
Kaiserslautern,0.0000,0.0189,0.2453,0.0000,0.2075,0.0189,0.0000,0.0189,0.2264,0.0189,...,0.1393,1.0319,49.442,7.760,101228,False,False,False,False,True
